# EMA Regime Filter Research

Evaluate whether an EMA-based regime filter improves the BOS breakout strategy.

- **trend_up**: close > EMA AND EMA_slope > 0 → allow LONG only
- **trend_down**: close < EMA AND EMA_slope < 0 → allow SHORT only
- **range**: all other cases → no trades

EMA_slope = EMA[t] - EMA[t-1]

Uses walk-forward framework: train / validation / OOS windows.

In [ ]:
import sys
from pathlib import Path

PROJECT_ROOT = Path.cwd()
while PROJECT_ROOT != PROJECT_ROOT.parent and not (PROJECT_ROOT / "data").exists():
    PROJECT_ROOT = PROJECT_ROOT.parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

import numpy as np
import pandas as pd
from collections import deque
from dataclasses import dataclass
from datetime import timedelta
from typing import Any, Dict, List, Optional, Tuple

from code.entry_exit import (
    Bar,
    BosSignal,
    ExitReason,
    PositionDirection,
    SameBarSlTpRule,
    SwingLevels,
    TakeProfitMode,
    TradeExit,
    TradePlan,
    check_exit_rules,
    detect_bos_signal,
    plan_trade_from_signal,
    update_last_swing_levels,
)
from code.risk import RiskConfig
from code.stop_loss import StopLossManager

## 1. Load Dataset

In [ ]:
DATA_PATH = PROJECT_ROOT / "data" / "processed" / "btc_15m.csv"
df = pd.read_csv(DATA_PATH)
df["open_time"] = pd.to_datetime(df["open_time"])
df = df.set_index("open_time")
df_ohlc = df[["open", "high", "low", "close"]].dropna()
print(f"Loaded {len(df_ohlc)} rows")
print(f"Date range: {df_ohlc.index.min()} to {df_ohlc.index.max()}")
df_ohlc.head()

## 2. Swing Detection (standalone, no QuantConnect)

In [ ]:
def swing_highs_lows_online(
    ohlc: pd.DataFrame,
    N_candidates: list = [5, 10, 20, 50],
    N_confirmation: int = 3,
    min_move_threshold: float = 0.0,
    min_bars_between_swings: int = 3,
) -> pd.DataFrame:
    """Online swing high/low detection (no repainting)."""
    swings = pd.DataFrame(index=ohlc.index, columns=["HighLow", "Level"], dtype=float)
    closes = ohlc["close"].values
    highs = ohlc["high"].values
    lows = ohlc["low"].values
    for N in N_candidates:
        window = N
        future_window = deque(maxlen=N_confirmation)
        last_swing_index = -min_bars_between_swings - 1
        for i in range(len(ohlc)):
            future_window.append(closes[i])
            if len(future_window) < N_confirmation:
                continue
            idx = i - N_confirmation
            if idx < 0 or (idx - last_swing_index) < min_bars_between_swings:
                continue
            left = max(0, idx - window)
            right = idx + 1
            candidate_close = closes[idx]
            window_values = closes[left:right]
            if candidate_close == max(window_values):
                if min_move_threshold == 0 or (candidate_close - min(window_values)) / candidate_close >= min_move_threshold:
                    swings.at[ohlc.index[idx], "HighLow"] = 1
                    swings.at[ohlc.index[idx], "Level"] = highs[idx]
                    last_swing_index = idx
            elif candidate_close == min(window_values):
                if min_move_threshold == 0 or (max(window_values) - candidate_close) / candidate_close >= min_move_threshold:
                    swings.at[ohlc.index[idx], "HighLow"] = -1
                    swings.at[ohlc.index[idx], "Level"] = lows[idx]
                    last_swing_index = idx
    return swings


SWING_PARAMS = {"N_candidates": [5, 10, 20, 50], "N_confirmation": 3, "min_move_threshold": 0.0, "min_bars_between_swings": 3}
swings = swing_highs_lows_online(df_ohlc[["close", "high", "low"]].copy(), **SWING_PARAMS)
print(f"Swings computed. Sample:")
swings.dropna().head(10)

## 3. EMA Regime Classification

In [ ]:
def compute_ema_regime(df: pd.DataFrame, ema_period: int) -> pd.Series:
    """
    Classify regime: trend_up, trend_down, or range.
    trend_up: close > EMA AND EMA_slope > 0
    trend_down: close < EMA AND EMA_slope < 0
    range: all other cases
    EMA_slope = EMA[t] - EMA[t-1]
    """
    ema = df["close"].ewm(span=ema_period, adjust=False).mean()
    ema_slope = ema.diff()  # EMA[t] - EMA[t-1]
    regime = pd.Series("range", index=df.index, dtype=object)
    regime[(df["close"] > ema) & (ema_slope > 0)] = "trend_up"
    regime[(df["close"] < ema) & (ema_slope < 0)] = "trend_down"
    return regime


def regime_allows_trade(regime: str, direction: PositionDirection) -> bool:
    """LONG allowed only in trend_up; SHORT only in trend_down."""
    if regime == "trend_up" and direction == PositionDirection.LONG:
        return True
    if regime == "trend_down" and direction == PositionDirection.SHORT:
        return True
    return False

## 4. Strategy Config & Backtest Engine

In [ ]:
@dataclass(frozen=True)
class StrategyConfig:
    sl_mode: str
    tp_mode: TakeProfitMode
    fixed_pct: float
    buffer_pct: float
    tp_mult: float
    same_bar_rule: SameBarSlTpRule


BASELINE_CONFIG = StrategyConfig(
    sl_mode="structural",
    tp_mode=TakeProfitMode.RR_BASED,
    fixed_pct=0.0025,
    buffer_pct=0.002,
    tp_mult=3.0,
    same_bar_rule=SameBarSlTpRule.WORST_CASE,
)

RISK = RiskConfig(risk_budget_cash=100.0)

In [ ]:
def _bars_from_df(df_ohlc: pd.DataFrame) -> List[Bar]:
    return [
        Bar(open=float(r["open"]), high=float(r["high"]), low=float(r["low"]), close=float(r["close"]), time=str(ts))
        for ts, r in df_ohlc.iterrows()
    ]


def compute_trade_metrics(trades: pd.DataFrame) -> Dict[str, Any]:
    """Full metrics: trades, win_rate, expectancy, profit_factor, sharpe, max_dd, avg_R."""
    if trades.empty:
        return {
            "number_of_trades": 0,
            "win_rate": float("nan"),
            "expectancy": float("nan"),
            "profit_factor": float("nan"),
            "sharpe_ratio": float("nan"),
            "max_drawdown": 0.0,
            "average_R_per_trade": float("nan"),
            "total_pnl": 0.0,
        }
    pnl = trades["pnl"].to_numpy(dtype=float)
    r_mult = trades["r_mult"].to_numpy(dtype=float)
    wins = pnl[pnl > 0]
    losses = pnl[pnl < 0]
    win_rate = float(np.mean(pnl > 0))
    expectancy = float(np.mean(pnl))
    avg_r = float(np.mean(r_mult))
    profit_factor = float("nan")
    if len(losses) > 0 and np.sum(losses) != 0:
        profit_factor = float(np.sum(wins) / abs(np.sum(losses)))
    cum = np.cumsum(pnl)
    peak = np.maximum.accumulate(cum)
    max_drawdown = float(np.min(cum - peak))
    sharpe_ratio = float("nan")
    if len(pnl) > 1 and np.std(pnl) > 0:
        sharpe_ratio = float(np.mean(pnl) / np.std(pnl) * np.sqrt(252 * 96))  # annualized, ~96 15m bars/day
    return {
        "number_of_trades": int(len(trades)),
        "win_rate": win_rate,
        "expectancy": expectancy,
        "profit_factor": profit_factor,
        "sharpe_ratio": sharpe_ratio,
        "max_drawdown": max_drawdown,
        "average_R_per_trade": avg_r,
        "total_pnl": float(np.sum(pnl)),
    }


def backtest_bos_breakout(
    df_ohlc: pd.DataFrame,
    swings: pd.DataFrame,
    config: StrategyConfig,
    risk: RiskConfig,
    regime_series: Optional[pd.Series] = None,
) -> Dict[str, Any]:
    """Run BOS breakout backtest. If regime_series provided, filter entries by regime."""
    df_ohlc = df_ohlc.dropna().copy()
    swings = swings.reindex(df_ohlc.index)
    bars = _bars_from_df(df_ohlc)
    sl_mgr = StopLossManager(mode=config.sl_mode, fixed_pct=config.fixed_pct, buffer_pct=config.buffer_pct)
    swing_levels = SwingLevels()
    last_valid_range = None
    state = "FLAT"
    trade_plan = None
    trades = []
    for i in range(len(bars)):
        ts = df_ohlc.index[i]
        bar = bars[i]
        hl = swings.iloc[i]["HighLow"] if pd.notna(swings.iloc[i].get("HighLow")) else np.nan
        lvl = swings.iloc[i]["Level"] if pd.notna(swings.iloc[i].get("Level")) else np.nan
        if pd.notna(hl) and pd.notna(lvl):
            swing_levels = update_last_swing_levels(swing_levels, highlow_flag=float(hl), level=float(lvl))
            if config.tp_mode == TakeProfitMode.RANGE_BASED:
                hi, lo = swing_levels.last_swing_high_price, swing_levels.last_swing_low_price
                if hi is not None and lo is not None and hi > lo:
                    last_valid_range = (float(hi), float(lo))
        if state in ("LONG", "SHORT") and trade_plan is not None:
            if i >= trade_plan.entry_candle_index:
                ex = check_exit_rules(bar=bar, direction=trade_plan.direction, sl_price=float(trade_plan.sl_price), tp_price=float(trade_plan.tp_price), same_bar_rule=config.same_bar_rule)
                if ex is not None:
                    entry_p = float(trade_plan.entry_price)
                    exit_p = float(ex.exit_price)
                    qty = float(trade_plan.quantity)
                    pnl = (exit_p - entry_p) * qty if trade_plan.direction == PositionDirection.LONG else (entry_p - exit_p) * qty
                    risk_per = abs(entry_p - float(trade_plan.sl_price))
                    r_mult = pnl / (risk_per * qty) if risk_per > 0 else float("nan")
                    trades.append({"entry_time": bars[trade_plan.entry_candle_index].time, "exit_time": bar.time, "direction": trade_plan.direction.value, "entry_price": entry_p, "exit_price": exit_p, "exit_reason": ex.exit_reason.value, "qty": qty, "pnl": float(pnl), "r_mult": r_mult})
                    state = "FLAT"
                    trade_plan = None
                    sl_mgr.reset()
                    continue
        if state != "FLAT":
            continue
        bos_signal = detect_bos_signal(bars=bars, t=i, swing_levels=swing_levels)
        if bos_signal is None:
            continue
        if regime_series is not None:
            regime_val = regime_series.iloc[i] if i < len(regime_series) else "range"
            if not regime_allows_trade(str(regime_val), bos_signal.direction):
                continue
        effective_swing = swing_levels
        if config.tp_mode == TakeProfitMode.RANGE_BASED and last_valid_range is not None:
            hi, lo = last_valid_range
            effective_swing = SwingLevels(last_swing_high_price=hi, last_swing_low_price=lo)
        sl_mgr.reset()
        try:
            plan = plan_trade_from_signal(bars=bars, bos_signal=bos_signal, swing_levels=effective_swing, stop_loss_manager=sl_mgr, tp_mode=config.tp_mode, tp_mult=config.tp_mult, risk_config=risk)
        except Exception:
            sl_mgr.reset()
            continue
        trade_plan = plan
        state = "LONG" if plan.direction == PositionDirection.LONG else "SHORT"
    trades_df = pd.DataFrame(trades)
    return {"trades": trades_df, "metrics": compute_trade_metrics(trades_df)}

## 5. Walk-Forward Evaluation

In [ ]:
def walk_forward_evaluation(
    df_ohlc: pd.DataFrame,
    swings: pd.DataFrame,
    config: StrategyConfig,
    risk: RiskConfig,
    ema_periods: List[int],
    is_window_days: int = 365,
    val_window_days: int = 90,
    oos_window_days: int = 90,
    anchored: bool = True,
) -> Dict[str, Any]:
    """
    Walk-forward: for each step, train | validation | OOS.
    For each ema_period (and baseline), evaluate on train/val/OOS.
    """
        start_ts = pd.Timestamp(df_ohlc.index.min())
    end_ts = pd.Timestamp(df_ohlc.index.max())
    train_end = start_ts + timedelta(days=is_window_days)
    all_results = []
    step_num = 0
    while True:
        val_end = train_end + timedelta(days=val_window_days)
        oos_end = val_end + timedelta(days=oos_window_days)
        if oos_end > end_ts:
            break
        train_df = df_ohlc.loc[start_ts:train_end].copy()
        val_df = df_ohlc.loc[train_end:val_end].copy()
        oos_df = df_ohlc.loc[val_end:oos_end].copy()
        if len(train_df) < 100 or len(val_df) < 20 or len(oos_df) < 20:
            train_end = train_end + timedelta(days=oos_window_days)
            continue
        train_sw = swings.reindex(train_df.index)
        val_sw = swings.reindex(val_df.index)
        oos_sw = swings.reindex(oos_df.index)
        for ema_p in [None] + ema_periods:
            regime_train = compute_ema_regime(train_df, ema_p) if ema_p else None
            regime_val = compute_ema_regime(val_df, ema_p) if ema_p else None
            regime_oos = compute_ema_regime(oos_df, ema_p) if ema_p else None
            res_train = backtest_bos_breakout(train_df, train_sw, config, risk, regime_train)
            res_val = backtest_bos_breakout(val_df, val_sw, config, risk, regime_val)
            res_oos = backtest_bos_breakout(oos_df, oos_sw, config, risk, regime_oos)
            all_results.append({
                "ema_period": ema_p if ema_p else "baseline",
                "step": step_num,
                "train_metrics": res_train["metrics"],
                "val_metrics": res_val["metrics"],
                "oos_metrics": res_oos["metrics"],
            })
        train_end = train_end + timedelta(days=oos_window_days)
        step_num += 1
    return {"results": all_results}

## 6. Run Hyperparameter Sweep & Walk-Forward

In [ ]:
# Train / Validation / OOS breakdown (averaged across steps)
def breakdown_by_window(wf_results):
    by_ema = {}
    for r in wf_results:
        ep = r["ema_period"]
        if ep not in by_ema:
            by_ema[ep] = {"train": [], "val": [], "oos": []}
        by_ema[ep]["train"].append(r["train_metrics"]["sharpe_ratio"])
        by_ema[ep]["val"].append(r["val_metrics"]["sharpe_ratio"])
        by_ema[ep]["oos"].append(r["oos_metrics"]["sharpe_ratio"])
    rows = []
    for ep, v in by_ema.items():
        rows.append({
            "ema_period": ep,
            "sharpe_train": np.nanmean(v["train"]),
            "sharpe_val": np.nanmean(v["val"]),
            "sharpe_oos": np.nanmean(v["oos"]),
        })
    return pd.DataFrame(rows)

breakdown = breakdown_by_window(wf["results"])
breakdown["_sort"] = breakdown["ema_period"].apply(lambda x: (0, 0) if x == "baseline" else (1, x))
breakdown = breakdown.sort_values("_sort").drop(columns=["_sort"])
display(breakdown)

In [ ]:
EMA_PERIODS = [100, 150, 200, 250, 300]
IS_DAYS = 365
VAL_DAYS = 90
OOS_DAYS = 90

wf = walk_forward_evaluation(
    df_ohlc,
    swings,
    BASELINE_CONFIG,
    RISK,
    ema_periods=EMA_PERIODS,
    is_window_days=IS_DAYS,
    val_window_days=VAL_DAYS,
    oos_window_days=OOS_DAYS,
    anchored=True,
)
print(f"Completed {len(wf['results'])} walk-forward steps across baseline + {len(EMA_PERIODS)} EMA periods")

## 7. Summary Table: ema_period | trades | sharpe | profit_factor | expectancy | max_dd

In [ ]:
def aggregate_by_ema(wf_results: List[Dict]) -> pd.DataFrame:
    """Aggregate metrics by ema_period across all walk-forward steps (OOS)."""
    by_ema = {}
    for r in wf_results:
        ep = r["ema_period"]
        m = r["oos_metrics"]
        if ep not in by_ema:
            by_ema[ep] = {"trades": [], "sharpe": [], "pf": [], "expectancy": [], "max_dd": [], "win_rate": [], "avg_r": []}
        by_ema[ep]["trades"].append(m["number_of_trades"])
        by_ema[ep]["sharpe"].append(m["sharpe_ratio"])
        by_ema[ep]["pf"].append(m["profit_factor"])
        by_ema[ep]["expectancy"].append(m["expectancy"])
        by_ema[ep]["max_dd"].append(m["max_drawdown"])
        by_ema[ep]["win_rate"].append(m["win_rate"])
        by_ema[ep]["avg_r"].append(m["average_R_per_trade"])
    rows = []
    for ep, v in by_ema.items():
        rows.append({
            "ema_period": ep,
            "trades": np.nanmean(v["trades"]),
            "sharpe": np.nanmean(v["sharpe"]),
            "profit_factor": np.nanmean(v["pf"]),
            "expectancy": np.nanmean(v["expectancy"]),
            "max_dd": np.nanmean(v["max_dd"]),
            "win_rate": np.nanmean(v["win_rate"]),
            "avg_R": np.nanmean(v["avg_r"]),
        })
    return pd.DataFrame(rows)


summary_oos = aggregate_by_ema(wf["results"])
summary_oos["_sort"] = summary_oos["ema_period"].apply(lambda x: (0, 0) if x == "baseline" else (1, x))
summary_oos = summary_oos.sort_values("_sort").drop(columns=["_sort"])
display(summary_oos[["ema_period", "trades", "sharpe", "profit_factor", "expectancy", "max_dd"]])

## 8. Regime Statistics (% time in trend_up, trend_down, range)

In [ ]:
# Dynamic analysis based on results
baseline = summary_oos[summary_oos["ema_period"] == "baseline"]
ema_df = summary_oos[summary_oos["ema_period"] != "baseline"]
if not baseline.empty and not ema_df.empty:
    b = baseline.iloc[0]
    best_sharpe = ema_df.loc[ema_df["sharpe"].idxmax()]
    best_pf = ema_df.loc[ema_df["profit_factor"].idxmax()]
    print("FINDINGS:")
    print("-" * 50)
    improves_sharpe = ema_df[ema_df["sharpe"] > b["sharpe"]]
    improves_pf = ema_df[ema_df["profit_factor"] > b["profit_factor"]]
    if len(improves_sharpe) > 0:
        print(f"EMA filter improves Sharpe vs baseline for: {list(improves_sharpe['ema_period'].values)}")
    else:
        print("No EMA period improves Sharpe vs baseline.")
    if len(improves_pf) > 0:
        print(f"EMA filter improves Profit Factor vs baseline for: {list(improves_pf['ema_period'].values)}")
    else:
        print("No EMA period improves Profit Factor vs baseline.")
    # Robust region: low std in stability
    stab_ema = stability[stability["ema_period"] != "baseline"]
    if not stab_ema.empty:
        robust = stab_ema.loc[stab_ema["sharpe_std"].idxmin()]
        print(f"Most stable (lowest sharpe_std): ema_period={robust['ema_period']}")

In [ ]:
regime_stats = []
for ema_p in EMA_PERIODS:
    regime = compute_ema_regime(df_ohlc, ema_p)
    n = len(regime)
    pct_up = (regime == "trend_up").sum() / n * 100 if n > 0 else 0
    pct_down = (regime == "trend_down").sum() / n * 100 if n > 0 else 0
    pct_range = (regime == "range").sum() / n * 100 if n > 0 else 0
    regime_stats.append({"ema_period": ema_p, "%_trend_up": round(pct_up, 1), "%_trend_down": round(pct_down, 1), "%_range": round(pct_range, 1)})

regime_df = pd.DataFrame(regime_stats)
display(regime_df)

## 9. Baseline vs EMA Filter Comparison

In [ ]:
baseline_row = summary_oos[summary_oos["ema_period"] == "baseline"].iloc[0]
ema_rows = summary_oos[summary_oos["ema_period"] != "baseline"]
print("=== BASELINE (no regime filter) ===")
print(f"  trades: {baseline_row['trades']:.1f}  sharpe: {baseline_row['sharpe']:.3f}  PF: {baseline_row['profit_factor']:.3f}  expectancy: {baseline_row['expectancy']:.2f}  max_dd: {baseline_row['max_dd']:.2f}")
print("\n=== EMA FILTER (OOS avg) ===")
for _, r in ema_rows.iterrows():
    print(f"  ema={r['ema_period']}: trades={r['trades']:.1f}  sharpe={r['sharpe']:.3f}  PF={r['profit_factor']:.3f}  expectancy={r['expectancy']:.2f}  max_dd={r['max_dd']:.2f}")

## 10. Stability Across Walk-Forward Windows

In [ ]:
def stability_table(wf_results: List[Dict]) -> pd.DataFrame:
    """Std of sharpe/profit_factor across steps per ema_period."""
    by_ema = {}
    for r in wf_results:
        ep = r["ema_period"]
        m = r["oos_metrics"]
        if ep not in by_ema:
            by_ema[ep] = {"sharpe": [], "pf": []}
        by_ema[ep]["sharpe"].append(m["sharpe_ratio"])
        by_ema[ep]["pf"].append(m["profit_factor"])
    rows = []
    for ep, v in by_ema.items():
        sh = np.array(v["sharpe"])
        pf = np.array(v["pf"])
        sh_valid = sh[~np.isnan(sh)]
        pf_valid = pf[~np.isnan(pf)]
        rows.append({
            "ema_period": ep,
            "sharpe_mean": np.nanmean(sh),
            "sharpe_std": np.std(sh_valid) if len(sh_valid) > 1 else 0,
            "pf_mean": np.nanmean(pf),
            "pf_std": np.std(pf_valid) if len(pf_valid) > 1 else 0,
        })
    return pd.DataFrame(rows)


stability = stability_table(wf["results"])
display(stability)

## 11. Analysis: Robust EMA Region

In [ ]:
print("""
ANALYSIS: EMA Regime Filter Evaluation
=====================================

1. Does the EMA filter improve the baseline?
   - Compare baseline OOS metrics vs each EMA period.
   - If sharpe/profit_factor/expectancy improve and max_dd reduces, the filter helps.

2. Which EMA region is robust?
   - Look for a plateau: multiple adjacent ema_periods with similar performance.
   - Avoid selecting a single 'best' - prefer a stable parameter region.
   - Check stability_table: lower std = more consistent across windows.

3. Is performance stable across walk-forward windows?
   - Low sharpe_std and pf_std indicate stability.
   - If one ema_period has high mean but high std, it may be overfit.

4. Regime statistics:
   - High %_range means fewer trades (filter is restrictive).
   - Balance: enough trend_up/trend_down for entries vs. filtering bad regimes.
""")